# 1. Project Setup
Canonical AI6132 Google Colab setup. Run every cell in order. Select **Runtime → Change runtime type → GPU** first; CPU setup remains supported. This notebook only prepares infrastructure: it downloads no datasets or models and runs no generation or experiments.

Edit the repository settings below. For a private repository, add a read-access token named `GITHUB_TOKEN` in Colab's Secrets panel and enable notebook access, then set `PRIVATE_REPOSITORY = True`. Never paste credentials into cells. Code stays in `/content`; persistent large files stay on Drive. Colab Free may disconnect, so rerun setup in a new session.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import tempfile

GITHUB_OWNER = "ZitingCCC"
GITHUB_REPO = "face-age-progression"
GITHUB_BRANCH = "main"
PRIVATE_REPOSITORY = False
PROJECT_ROOT = Path("/content/face-age-progression")
DRIVE_ROOT = Path("/content/drive/MyDrive/AI6132")
# Set before any CUDA initialization; the configured seed is applied later.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


## 2. Runtime Information
These diagnostics inspect the supplied runtime without replacing packages. Missing torchvision is acceptable; an incompatible installation is reported. CUDA absence does not prevent setup.


In [ ]:
import platform
print("Python:", sys.version)
print("Platform:", platform.platform())
try:
    import torch
except ImportError:
    torch = None
    print("PyTorch is not installed; use a standard Colab runtime.")
else:
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("PyTorch CUDA:", torch.version.cuda)
try:
    import torchvision
    print("torchvision:", torchvision.__version__)
except Exception as error:
    print("torchvision unavailable:", type(error).__name__)


## 3. Google Drive
Approve the Drive authorization prompt. Mounting reuses an existing mount. Directory creation preserves existing data, models, outputs, and caches.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
for folder in ("data", "models", "generated", "results", "cache"):
    (DRIVE_ROOT / folder).mkdir(parents=True, exist_ok=True)
print("Persistent storage:", DRIVE_ROOT)


## 4. Repository Setup
Clone once using HTTPS. Private authentication uses a temporary Git askpass helper and an environment variable; the helper contains no token and is removed afterward. Credentials are never put in remote URLs, command arguments, or Drive files. Git output is captured and failures show only the operation, avoiding credential-bearing diagnostics.

An existing checkout must have the expected origin and no tracked or untracked local changes. Setup fetches the requested branch and permits only a fast-forward update. If setup stops, inspect and preserve your work manually before rerunning; it never resets, cleans, or discards files. A locally divergent branch also stops.


In [ ]:
import re
if not all(re.fullmatch(r"[A-Za-z0-9_.-]+", value) for value in (GITHUB_OWNER, GITHUB_REPO)):
    raise ValueError("Use a valid GitHub owner and repository name.")
if subprocess.run(["git", "check-ref-format", "--branch", GITHUB_BRANCH],
                  capture_output=True).returncode:
    raise ValueError("Invalid branch name.")
repository_url = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"

def setup_repository():
    token = None
    git_env = os.environ.copy()
    git_env.update(GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                   GIT_CONFIG_KEY_0="credential.helper", GIT_CONFIG_VALUE_0="")
    def git(*args, cwd=None):
        result = subprocess.run(["git", *args], cwd=cwd, env=git_env,
                                capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError(f"Git {args[0]} failed. Check repository access, branch, or local divergence.")
        return result.stdout.strip()
    with tempfile.TemporaryDirectory(prefix="ai6132-git-", dir="/content") as temporary:
        try:
            if PRIVATE_REPOSITORY:
                from google.colab import userdata
                try:
                    token = userdata.get("GITHUB_TOKEN")
                except Exception:
                    raise RuntimeError("Enable access to GITHUB_TOKEN in Colab Secrets.") from None
                if not token:
                    raise RuntimeError("GITHUB_TOKEN is empty.")
                helper = Path(temporary) / "askpass.py"
                helper.write_text("#!/usr/bin/env python3\nimport os, sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['AI6132_GIT_TOKEN'])\n")
                helper.chmod(0o700)
                git_env.update(GIT_ASKPASS=str(helper), AI6132_GIT_TOKEN=token)
            if not PROJECT_ROOT.exists():
                git("clone", "--branch", GITHUB_BRANCH, "--single-branch", repository_url, str(PROJECT_ROOT))
            else:
                if not (PROJECT_ROOT / ".git").exists():
                    raise RuntimeError("PROJECT_ROOT exists but is not a Git checkout.")
                if Path(git("rev-parse", "--show-toplevel", cwd=PROJECT_ROOT)).resolve() != PROJECT_ROOT.resolve():
                    raise RuntimeError("PROJECT_ROOT is not the repository root.")
                if git("remote", "get-url", "origin", cwd=PROJECT_ROOT) != repository_url:
                    raise RuntimeError("Origin differs from the configured HTTPS repository; inspect it manually.")
                if git("status", "--porcelain", "--untracked-files=all", cwd=PROJECT_ROOT):
                    raise RuntimeError("Local changes exist. Preserve or commit them before rerunning setup.")
                git("fetch", "origin", f"+refs/heads/{GITHUB_BRANCH}:refs/remotes/origin/{GITHUB_BRANCH}", cwd=PROJECT_ROOT)
                git("checkout", GITHUB_BRANCH, cwd=PROJECT_ROOT)
                git("merge", "--ff-only", f"origin/{GITHUB_BRANCH}", cwd=PROJECT_ROOT)
            print("Repository HEAD:", git("rev-parse", "HEAD", cwd=PROJECT_ROOT))
        finally:
            git_env.pop("AI6132_GIT_TOKEN", None)
            token = None

setup_repository()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


## 5. Dependency Installation
Install `requirements.txt` without a blanket upgrade. Exact constraints preserve the runtime's installed PyTorch and torchvision, including their CUDA build suffixes. If requirements conflict, installation stops instead of replacing the supplied builds. Use a fresh standard Colab runtime to resolve an incompatible environment. If pip requests a runtime restart, restart and rerun from the top.


In [ ]:
from importlib.metadata import version, PackageNotFoundError
try:
    torch_version_before = version("torch")
except PackageNotFoundError:
    raise RuntimeError("PyTorch is missing. Select a standard Colab runtime before installing dependencies.") from None
constraints = [f"torch=={torch_version_before}"]
try:
    constraints.append(f"torchvision=={version('torchvision')}")
except PackageNotFoundError:
    pass
with tempfile.TemporaryDirectory(prefix="ai6132-pip-", dir="/content") as temporary:
    constraint_path = Path(temporary) / "constraints.txt"
    constraint_path.write_text("\n".join(constraints) + "\n")
    subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                    str(PROJECT_ROOT / "requirements.txt"), "-c", str(constraint_path)], check=True)
assert version("torch") == torch_version_before, "PyTorch installation changed unexpectedly"
import torch


## 6. GPU Diagnostics
Check the actual GPU assigned to this session; Colab Free does not guarantee a particular device. CPU setup is valid, but generation needs a suitable GPU runtime.


In [ ]:
from src.utils.device import detect_device, device_info
DEVICE = detect_device()
DEVICE_INFO = device_info()
print("PyTorch CUDA:", torch.version.cuda)
if torch.cuda.is_available():
    gpu_index = torch.cuda.current_device()
    properties = torch.cuda.get_device_properties(gpu_index)
    print("GPU model:", properties.name)
    print(f"Total GPU VRAM: {properties.total_memory / 10**9:.2f} GB")
    print(f"Allocated GPU memory: {torch.cuda.memory_allocated(gpu_index) / 10**9:.3f} GB")
    print(f"Reserved GPU memory: {torch.cuda.memory_reserved(gpu_index) / 10**9:.3f} GB")
else:
    print("CUDA GPU is not available. Setup can continue, but generation experiments should not be run in this runtime.")


## 7. Persistent Storage
Reuse Task 01's path resolver for the checkout and Drive storage root. Its legacy `datasets`/`outputs` properties use a different layout, so the explicit paths below define the Task 02 layout. Future workflows should use these roots for large files. Creating directories never clears their contents.

Hugging Face cache variables are set before any future Hugging Face imports. This cell downloads nothing and needs no Hugging Face token. If those libraries were already imported in this session, restart and run setup first so they see these settings.


In [ ]:
from src.utils.paths import configure_paths
PATHS = configure_paths("drive", project_root=PROJECT_ROOT, drive_root=DRIVE_ROOT)
PROJECT_ROOT = PATHS.project
DRIVE_ROOT = PATHS.storage
DATA_ROOT = DRIVE_ROOT / "data"
MODEL_ROOT = DRIVE_ROOT / "models"
GENERATED_ROOT = DRIVE_ROOT / "generated"
RESULTS_ROOT = DRIVE_ROOT / "results"
CACHE_ROOT = DRIVE_ROOT / "cache"
for root in (DATA_ROOT, MODEL_ROOT, GENERATED_ROOT, RESULTS_ROOT, CACHE_ROOT):
    root.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["HF_HUB_CACHE"] = str(CACHE_ROOT / "huggingface" / "hub")
os.environ["HF_DATASETS_CACHE"] = str(CACHE_ROOT / "huggingface" / "datasets")
for name in ("HF_HOME", "HF_HUB_CACHE", "HF_DATASETS_CACHE"):
    Path(os.environ[name]).mkdir(parents=True, exist_ok=True)
for name in ("PROJECT_ROOT", "DRIVE_ROOT", "DATA_ROOT", "MODEL_ROOT", "GENERATED_ROOT", "RESULTS_ROOT", "CACHE_ROOT"):
    print(f"{name}: {globals()[name]}")


## 8. Project Configuration
Load all three YAML files using the existing utilities. Apply the experiment seed and deterministic setting. CUDA workspace configuration was set before runtime diagnostics. Reproducibility is limited to a fixed software/hardware environment.

`SESSION_CONFIG` records the session's Drive path override in memory; repository YAML files are preserved. No experiment is run.


In [ ]:
from src.utils.config import load_config
from src.utils.reproducibility import set_seed
EXPERIMENT_CONFIG = load_config(PROJECT_ROOT / "configs/experiment.yaml")
DIFFUSION_CONFIG = load_config(PROJECT_ROOT / "configs/diffusion.yaml")
IDENTITY_CONFIG = load_config(PROJECT_ROOT / "configs/identity.yaml")
set_seed(EXPERIMENT_CONFIG["seed"], EXPERIMENT_CONFIG["deterministic"])
SESSION_CONFIG = dict(EXPERIMENT_CONFIG)
SESSION_CONFIG["paths"] = dict(environment="drive", project_root=str(PROJECT_ROOT),
                               drive_root=str(DRIVE_ROOT), storage_root=str(DRIVE_ROOT))
print("Loaded experiment, diffusion, and identity configurations.")
print("Seed:", EXPERIMENT_CONFIG["seed"], "Device:", DEVICE)


## 9. Import Smoke Test
This is infrastructure validation, not an experimental result. A unique temporary directory under Drive's cache tests metadata serialization and is removed on successful completion or ordinary failure. Interrupted sessions may leave a small `setup-smoke-*` directory; no existing files are removed by later runs.


In [ ]:
import random
import numpy as np
import src
from src.utils.metadata import ExperimentMetadata, write_metadata, read_metadata
from src.utils.resume import is_completed

assert all(isinstance(config, dict) for config in (EXPERIMENT_CONFIG, DIFFUSION_CONFIG, IDENTITY_CONFIG))
assert all(root.is_dir() for root in (DATA_ROOT, MODEL_ROOT, GENERATED_ROOT, RESULTS_ROOT, CACHE_ROOT))
seed = EXPERIMENT_CONFIG["seed"]
deterministic = EXPERIMENT_CONFIG["deterministic"]
set_seed(seed, deterministic)
first = (random.random(), float(np.random.rand()), torch.rand(3))
set_seed(seed, deterministic)
second = (random.random(), float(np.random.rand()), torch.rand(3))
assert first[:2] == second[:2] and torch.equal(first[2], second[2])
assert DEVICE.type == ("cuda" if DEVICE_INFO["cuda_available"] else "cpu")
with tempfile.TemporaryDirectory(prefix="setup-smoke-", dir=CACHE_ROOT) as temporary:
    temporary_root = Path(temporary)
    metadata_path = temporary_root / "setup-check.json"
    record = ExperimentMetadata(
        experiment_id="setup-smoke-test", method="infrastructure-only", model=None,
        source_image="", source_age=None, target_age=None, seed=seed,
        configuration={"purpose": "temporary setup validation"}, device=DEVICE_INFO,
        status="pending")
    write_metadata(metadata_path, record)
    restored = read_metadata(metadata_path)
    assert restored["experiment_id"] == "setup-smoke-test" and restored["status"] == "pending"
    assert restored["configuration"] == record.configuration
    assert not is_completed(temporary_root / "nonexistent-output", metadata_path)
set_seed(seed, deterministic)
SMOKE_TEST_PASSED = True
print("Setup smoke test passed; temporary metadata removed. No experiment was run.")


## 10. Setup Summary
Confirm the smoke test passed, the printed CUDA diagnostics match your runtime, and the persistent roots point to `MyDrive/AI6132`. This establishes setup only. Drive mounting, GPU operation, and full top-to-bottom execution must be verified in real Colab; CPU development checks cannot establish them.

On disconnection, open a fresh runtime and rerun. Preserve local code edits in Git before updating an existing checkout. Future experiments must save each output and metadata immediately to Drive and use resume checks; none are implemented here.


In [ ]:
assert SMOKE_TEST_PASSED
print("Setup ready:", PROJECT_ROOT)
print("Persistent storage:", DRIVE_ROOT)
print("Selected device:", DEVICE)
print("Smoke test: passed")
print("Datasets/models downloaded: none. Faces generated: none. Experimental results: none.")
